# Pipeline Execution Driver

목적: 전체 분석 workflow를 단계별로 실행하면서 로그, 진행도, 산출물 상태를 남깁니다.

Workflow:
1. LIF to TIFF parsing: `lif_file_analysis_fin.ipynb`
2. Denoise preprocessing: `02_denoise_preprocessing.ipynb`
3. TIFF Z-stack registration: `rigid_registration.ipynb`
4. ROI selection: `ROI_selection_04.py`
5. Signal intensity: `05_signal_intensity_analysis.ipynb`
6. SNR: `06_snr_analysis.ipynb`
7. 3D particle/object morphology: `07_particle_object_morphology_3d.ipynb`
8. Reference:Target colocalization: `08_reference_target_colocalization.ipynb`

이 driver는 결과 해석보다는 실행 기록과 산출물 생성 상태 확인에 집중합니다.


## 0. Runtime Configuration


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
LOG_DIR = PROJECT_DIR / "logs"
EXECUTED_NOTEBOOK_DIR = PROJECT_DIR / "executed_notebooks"

LOG_DIR.mkdir(exist_ok=True)
EXECUTED_NOTEBOOK_DIR.mkdir(exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
JSONL_LOG = LOG_DIR / f"pipeline_run_{RUN_ID}.jsonl"

STAGE_FILES = {
    "01_lif_to_tiff": PROJECT_DIR / "01_lif_to_tiff.ipynb",
    "02_denoise_preprocessing": PROJECT_DIR / "02_denoise_preprocessing.ipynb",
    "03_registration": PROJECT_DIR / "03_rigid_registration.ipynb",
    "04_roi_selection": PROJECT_DIR / "ROI_selection_04.py",
    "05_signal_intensity": PROJECT_DIR / "05_signal_intensity_analysis.ipynb",
    "06_snr": PROJECT_DIR / "06_snr_analysis.ipynb",
    "07_morphology_3d": PROJECT_DIR / "07_particle_object_morphology_3d.ipynb",
    "08_colocalization": PROJECT_DIR / "08_reference_target_colocalization.ipynb",
}

# Toggle individual stages here. Keep False while editing notebooks.
RUN_STEPS = {
    "01_lif_to_tiff": False,
    "02_denoise_preprocessing": False,
    "03_registration": False,
    "04_roi_selection": False,
    "05_signal_intensity": False,
    "06_snr": False,
    "07_morphology_3d": False,
    "08_colocalization": False,
}

CHANNELS = ("DAPI", "Reference", "Target")
REQUIRED_STACK_FILES = tuple(f"{ch}_stack.tif" for ch in CHANNELS)
REQUIRED_REGISTERED_FILES = tuple(f"{ch}_stack_registered.tif" for ch in CHANNELS)

# Test-set controls used by this driver for inspection/progress/QC checks.
# Notebook execution still runs the selected stage notebook as written; use these
# controls while designing code to inspect a small subset before enabling full runs.
TEST_MODE = True
SELECT_LIF_NAMES = None       # e.g. ["1_PSD Ms MA1046_Homer Rb SYSY.lif"] or None
SELECT_SERIES_CONTAINS = "63x" # substring filter for series folders/header names; None disables
SELECT_SERIES_NAMES = None    # exact series-folder names; None disables
MAX_TEST_SERIES = 3 if TEST_MODE else None

print({
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "run_id": RUN_ID,
    "jsonl_log": str(JSONL_LOG),
    "test_mode": TEST_MODE,
    "select_lif_names": SELECT_LIF_NAMES,
    "select_series_contains": SELECT_SERIES_CONTAINS,
    "select_series_names": SELECT_SERIES_NAMES,
    "max_test_series": MAX_TEST_SERIES,
    "stage_files": {k: str(v) for k, v in STAGE_FILES.items()},
})


## 1. Logging Helpers


In [ ]:
def log_event(step: str, status: str, **payload):
    event = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "run_id": RUN_ID,
        "step": step,
        "status": status,
        **payload,
    }
    with open(JSONL_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False, default=str) + "\n")
    print(event)
    return event


def execute_notebook(step: str, notebook_path: Path, timeout: int | None = None):
    notebook_path = Path(notebook_path)
    output_name = f"{notebook_path.stem}__executed_{RUN_ID}.ipynb"
    output_path = EXECUTED_NOTEBOOK_DIR / output_name
    text_log_path = LOG_DIR / f"{step}_{RUN_ID}.txt"

    if not notebook_path.exists():
        log_event(step, "missing", notebook=str(notebook_path))
        return {"step": step, "ok": False, "reason": "missing", "output": None}

    cmd = [
        sys.executable,
        "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute", str(notebook_path),
        "--output", str(output_path),
        "--ExecutePreprocessor.timeout=-1" if timeout is None else f"--ExecutePreprocessor.timeout={timeout}",
    ]

    log_event(step, "started", command=" ".join(cmd), notebook=str(notebook_path))
    proc = subprocess.run(
        cmd,
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
    )
    text_log_path.write_text(proc.stdout + "\n\nSTDERR:\n" + proc.stderr, encoding="utf-8")
    status = "completed" if proc.returncode == 0 else "failed"
    log_event(
        step,
        status,
        returncode=proc.returncode,
        executed_notebook=str(output_path),
        text_log=str(text_log_path),
    )
    return {
        "step": step,
        "ok": proc.returncode == 0,
        "returncode": proc.returncode,
        "output": output_path,
        "text_log": text_log_path,
    }


def execute_stage(step: str, stage_path: Path, timeout: int | None = None):
    stage_path = Path(stage_path)
    if stage_path.suffix.lower() == ".ipynb":
        return execute_notebook(step, stage_path, timeout=timeout)
    if stage_path.suffix.lower() == ".py":
        if not stage_path.exists():
            log_event(step, "missing", file=str(stage_path))
            return {"step": step, "ok": False, "reason": "missing", "output": None}
        command = f"{sys.executable} {stage_path.name}"
        log_event(
            step,
            "manual_python_stage",
            file=str(stage_path),
            command=command,
            reason="Interactive ROI selection needs a GUI session; run it directly from a terminal.",
        )
        return {
            "step": step,
            "ok": True,
            "reason": "manual_python_stage",
            "command": command,
            "output": stage_path,
        }
    if stage_path.suffix.lower() == ".m":
        if not stage_path.exists():
            log_event(step, "missing", file=str(stage_path))
            return {"step": step, "ok": False, "reason": "missing", "output": None}
        log_event(
            step,
            "manual_matlab_stage",
            file=str(stage_path),
            reason="Legacy MATLAB ROI selection stage; prefer ROI_selection_04.py.",
        )
        return {
            "step": step,
            "ok": True,
            "reason": "manual_matlab_stage",
            "output": stage_path,
        }
    log_event(step, "unsupported_stage_type", file=str(stage_path))
    return {"step": step, "ok": False, "reason": "unsupported_stage_type", "output": None}


## 2. Input / Output Inventory


In [ ]:
def count_paths(pattern: str):
    return len(list(PROJECT_DIR.glob(pattern)))


def discover_lif_files(data_dir: Path = DATA_DIR):
    lif_paths = sorted(data_dir.glob("*.lif")) if data_dir.exists() else []
    if SELECT_LIF_NAMES:
        allowed = set(SELECT_LIF_NAMES)
        lif_paths = [p for p in lif_paths if p.name in allowed or p.stem in allowed]
    return lif_paths


def _series_passes_filter(lif_name: str, series_name: str):
    if SELECT_LIF_NAMES:
        allowed = set(SELECT_LIF_NAMES)
        if lif_name not in allowed and f"{lif_name}.lif" not in allowed:
            return False
    if SELECT_SERIES_NAMES and series_name not in set(SELECT_SERIES_NAMES):
        return False
    if SELECT_SERIES_CONTAINS and SELECT_SERIES_CONTAINS.lower() not in series_name.lower():
        return False
    return True


def apply_series_limit(df: pd.DataFrame):
    if df.empty:
        return df
    if MAX_TEST_SERIES is None:
        return df.reset_index(drop=True)
    return df.head(int(MAX_TEST_SERIES)).reset_index(drop=True)


def discover_stack_series(data_dir: Path = DATA_DIR, apply_filter: bool = False):
    rows = []
    if not data_dir.exists():
        return pd.DataFrame(rows)
    for stacks_dir in sorted(data_dir.glob("*/*/stacks")):
        series_dir = stacks_dir.parent
        lif_name = series_dir.parent.name
        series_name = series_dir.name
        if apply_filter and not _series_passes_filter(lif_name, series_name):
            continue
        reg_dir = series_dir / "registered_stacks"
        rows.append({
            "lif_name": lif_name,
            "series_name": series_name,
            "series_dir": str(series_dir),
            "stacks_dir": str(stacks_dir),
            "has_dapi": (stacks_dir / "DAPI_stack.tif").exists(),
            "has_reference": (stacks_dir / "Reference_stack.tif").exists(),
            "has_target": (stacks_dir / "Target_stack.tif").exists(),
            "has_all_raw_stacks": all((stacks_dir / name).exists() for name in REQUIRED_STACK_FILES),
            "registered_tiff_count": len(list(reg_dir.glob("*_registered.tif"))) if reg_dir.exists() else 0,
            "has_metadata": (series_dir / "metadata.json").exists(),
        })
    df = pd.DataFrame(rows)
    return apply_series_limit(df) if apply_filter else df


def inventory():
    stack_df = discover_stack_series()
    selected_df = discover_stack_series(apply_filter=True)
    rows = [
        {"item": "LIF files selected", "count": len(discover_lif_files())},
        {"item": "stack folders total", "count": len(stack_df)},
        {"item": "stack folders selected/test", "count": len(selected_df)},
        {"item": "complete raw stack sets total", "count": int(stack_df["has_all_raw_stacks"].sum()) if not stack_df.empty else 0},
        {"item": "complete raw stack sets selected/test", "count": int(selected_df["has_all_raw_stacks"].sum()) if not selected_df.empty else 0},
        {"item": "DAPI stacks", "count": count_paths("data/*/*/stacks/DAPI_stack.tif")},
        {"item": "Reference stacks", "count": count_paths("data/*/*/stacks/Reference_stack.tif")},
        {"item": "Target stacks", "count": count_paths("data/*/*/stacks/Target_stack.tif")},
        {"item": "registered stack folders", "count": len(list(DATA_DIR.glob("*/*/registered_stacks"))) if DATA_DIR.exists() else 0},
        {"item": "registered TIFFs", "count": count_paths("data/*/*/registered_stacks/*_registered.tif")},
        {"item": "metadata.json", "count": count_paths("data/*/*/metadata.json")},
    ]
    return pd.DataFrame(rows)


inv = inventory()
display(inv)
log_event("inventory", "completed", rows=inv.to_dict(orient="records"))

stack_series_df = discover_stack_series()
selected_stack_series_df = discover_stack_series(apply_filter=True)
print("All stack series:")
display(stack_series_df)
print("Selected/test stack series:")
display(selected_stack_series_df)

if not selected_stack_series_df.empty:
    incomplete_raw_df = selected_stack_series_df[~selected_stack_series_df["has_all_raw_stacks"]]
    if not incomplete_raw_df.empty:
        display(incomplete_raw_df)
        log_event("lif_to_tiff_stack_check", "incomplete", rows=incomplete_raw_df.to_dict(orient="records"))
    else:
        log_event("lif_to_tiff_stack_check", "completed", n_complete_raw_stack_sets=len(selected_stack_series_df))


## 2.5 LIF Header / Series Inspection

`01_lif_to_tiff.ipynb`에서 수행하던 LIF metadata inspection을 driver에서도 요약 확인합니다. 실제 TIFF 추출은 stage 실행 셀에서 수행합니다.


In [ ]:
def inspect_lif_headers(lif_paths=None, verbose=False):
    try:
        from readlif.reader import LifFile
    except Exception as exc:
        log_event("lif_header_inspection", "skipped", reason=f"readlif import failed: {exc}")
        return pd.DataFrame()

    rows = []
    for lif_path in lif_paths or discover_lif_files():
        try:
            lif = LifFile(str(lif_path))
            for idx in range(lif.num_images):
                img = lif.get_image(idx)
                series_name = str(img.name)
                if not _series_passes_filter(lif_path.stem, series_name):
                    continue
                row = {
                    "lif_file": lif_path.name,
                    "lif_path": str(lif_path),
                    "series_index": idx,
                    "series_name": series_name,
                    "x_size": getattr(img.dims, "x", None),
                    "y_size": getattr(img.dims, "y", None),
                    "z_slices": getattr(img.dims, "z", None),
                    "t_frames": getattr(img.dims, "t", None),
                    "channels": getattr(img, "channels", None),
                    "scale": str(getattr(img, "scale", None)),
                    "matches_63x": "63x" in series_name.lower(),
                }
                rows.append(row)
                if verbose:
                    print(row)
        except Exception as exc:
            rows.append({"lif_file": lif_path.name, "lif_path": str(lif_path), "error": str(exc)})
    df = pd.DataFrame(rows)
    df = apply_series_limit(df) if not df.empty else df
    if not df.empty:
        display(df)
        summary = df.groupby("lif_file").agg(
            n_series=("series_index", "count"),
            n_63x=("matches_63x", "sum"),
        ).reset_index() if "matches_63x" in df.columns else pd.DataFrame()
        if not summary.empty:
            display(summary)
        log_event("lif_header_inspection", "completed", n_rows=len(df))
    return df


lif_header_df = inspect_lif_headers()


## 3. Execute Selected Steps

`RUN_STEPS`는 notebook stage를 실행하고, ROI Python stage는 interactive manual step으로 로그만 남깁니다. `ROI_selection_04.py`는 터미널에서 직접 실행하세요.


In [ ]:
run_results = []
for step, should_run in RUN_STEPS.items():
    if not should_run:
        log_event(step, "skipped", reason="RUN_STEPS disabled")
        continue
    run_results.append(execute_stage(step, STAGE_FILES[step]))

run_results_df = pd.DataFrame(run_results)
display(run_results_df)


## 4. Post-run Output Check


In [ ]:
def series_output_status(apply_filter: bool = False):
    rows = []
    if not DATA_DIR.exists():
        return pd.DataFrame(rows)
    for stacks_dir in sorted(DATA_DIR.glob("*/*/stacks")):
        series_dir = stacks_dir.parent
        lif_name = series_dir.parent.name
        series_name = series_dir.name
        if apply_filter and not _series_passes_filter(lif_name, series_name):
            continue
        reg_dir = series_dir / "registered_stacks"
        metadata_path = series_dir / "metadata.json"
        raw_stack_paths = {ch: stacks_dir / f"{ch}_stack.tif" for ch in CHANNELS}
        registered_paths = {ch: reg_dir / f"{ch}_stack_registered.tif" for ch in CHANNELS}
        rows.append({
            "lif_name": lif_name,
            "series_name": series_name,
            "has_dapi": raw_stack_paths["DAPI"].exists(),
            "has_reference": raw_stack_paths["Reference"].exists(),
            "has_target": raw_stack_paths["Target"].exists(),
            "raw_stack_count": sum(path.exists() for path in raw_stack_paths.values()),
            "has_all_raw_stacks": all(path.exists() for path in raw_stack_paths.values()),
            "has_registered_dir": reg_dir.exists(),
            "registered_tiff_count": sum(path.exists() for path in registered_paths.values()),
            "has_all_registered_stacks": all(path.exists() for path in registered_paths.values()),
            "has_metadata": metadata_path.exists(),
        })
    df = pd.DataFrame(rows)
    return apply_series_limit(df) if apply_filter else df


status_df = series_output_status()
selected_status_df = series_output_status(apply_filter=True)
print("All series output status:")
display(status_df)
print("Selected/test series output status:")
display(selected_status_df)
if not selected_status_df.empty:
    summary = selected_status_df.groupby("lif_name").agg(
        n_series=("series_name", "count"),
        n_complete_raw=("has_all_raw_stacks", "sum"),
        n_complete_registered=("has_all_registered_stacks", "sum"),
    ).reset_index()
    display(summary)
    log_event("post_run_output_check", "completed", summary=summary.to_dict(orient="records"))


## 5. Registration / Preprocessing Metadata Summary


In [ ]:
def load_metadata_summary():
    rows = []
    for p in sorted(DATA_DIR.glob("*/*/metadata.json")) if DATA_DIR.exists() else []:
        if p.name.startswith("._"):
            continue
        try:
            meta = json.loads(p.read_text(encoding="utf-8"))
        except Exception as exc:
            rows.append({"metadata_path": str(p), "error": str(exc)})
            continue
        reg = meta.get("registration") or meta.get("elastix_registration") or {}
        prep = meta.get("preprocessing") or {}
        rows.append({
            "lif_name": meta.get("lif_name", p.parent.parent.name),
            "series_name": meta.get("series_name", p.parent.name),
            "z_slices": meta.get("z_slices"),
            "registration_method": reg.get("method"),
            "final_z_slices": reg.get("final_z_slices"),
            "clipped": reg.get("clipped_due_to_failure", reg.get("clipped_at") is not None if reg else None),
            "jump_threshold": reg.get("jump_threshold_voxels"),
            "fatal_threshold": reg.get("fatal_jump_threshold", reg.get("fatal_jump_threshold_voxels")),
            "denoise_lines": prep.get("denoise_lines"),
            "metadata_path": str(p),
        })
    return pd.DataFrame(rows)


metadata_summary_df = load_metadata_summary()
display(metadata_summary_df)
if not metadata_summary_df.empty:
    log_event("metadata_summary", "completed", n_rows=len(metadata_summary_df))
